# 02b — Traduction Google Texte avec Playwright

Ce notebook pilote un navigateur **visible** pour traduire progressivement les partitions français → baoulé dans le mode Texte. Google utilise le code `bci` dans l'URL de son interface pour le baoulé. Le pilote de 10 blocs a été validé avant l'activation du mode complet.

Principes de sécurité et de reproductibilité : une seule session, débit fixe et modéré, sauvegarde après chaque bloc, reprise automatique, aucune rotation de proxy ou d'identité, aucun contournement de CAPTCHA, et arrêt immédiat si Google affiche une protection.

> Cette interface grand public n'est pas une API. Vérifiez que cet usage respecte les conditions applicables à votre organisation avant de dépasser le pilote.

In [1]:
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import asyncio
import hashlib
import json
import os
import re
import subprocess
import sys
import threading
import time
from urllib.parse import urlencode

PROJECT_ROOT = Path.cwd()
SPLIT_DIR = PROJECT_ROOT / "data" / "01_splits"
STEP_DIR = PROJECT_ROOT / "data" / "02b_text_browser"
STEP_DIR.mkdir(parents=True, exist_ok=True)

UNITS_PATH = STEP_DIR / "translation_units.jsonl"
BATCHES_PATH = STEP_DIR / "translation_batches.jsonl"
RESULTS_PATH = STEP_DIR / "browser_results.jsonl"
PARTIAL_TRANSLATIONS_PATH = STEP_DIR / "partial_translations.jsonl"
LOCAL_BROWSER_LIB_DIR = PROJECT_ROOT / ".playwright-libs" / "usr" / "lib" / "x86_64-linux-gnu"

TARGET_LANGUAGE_CODE = "bci"
MAX_BATCH_CHARACTERS = 3_500  # garde aussi l'URL encodée sous une taille fiable
BATCHES_PER_SESSION = 100     # relancer la cellule pour traiter la session suivante
DELAY_SECONDS = 8.0           # délai fixe entre deux blocs
TRANSLATION_TIMEOUT_SECONDS = 45
RUN_BROWSER = True           # passer à True après lecture de la procédure
BROWSER_CHANNEL = None     # utiliser le Chromium installé et géré par Playwright

print("Répertoire de travail :", STEP_DIR)
print("Code cible Google     :", TARGET_LANGUAGE_CODE)
print("Blocs maximum par session :", BATCHES_PER_SESSION)

Répertoire de travail : /home/vedem_worker/scrapt-bau/data/02b_text_browser
Code cible Google     : bci
Blocs maximum par session : 100


## 1. Construire les unités de traduction

Pour `fr_bau`, seule la réponse est traduite. Pour `bau_bau`, l'instruction, le contexte non vide et la réponse sont traduits. Les textes exceptionnellement longs sont découpés à une frontière naturelle puis réassemblés après traduction.

In [2]:
def read_jsonl(path):
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def write_jsonl_atomic(path, records):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="\n") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    temporary.replace(path)

FIELD_MAP = {
    "fr_bau": [("output_fr", "output_bau")],
    "bau_bau": [
        ("instruction_fr", "instruction_bau"),
        ("input_fr", "input_bau"),
        ("output_fr", "output_bau"),
    ],
}

source_partitions = {
    direction: read_jsonl(SPLIT_DIR / f"{direction}_source.jsonl")
    for direction in FIELD_MAP
}

def split_long_text(text, max_chars):
    if len(text) <= max_chars:
        return [text]
    chunks = []
    remaining = text
    while len(remaining) > max_chars:
        window = remaining[:max_chars + 1]
        candidates = [
            window.rfind("\n\n"),
            window.rfind("\n"),
            window.rfind(". "),
            window.rfind("! "),
            window.rfind("? "),
            window.rfind("; "),
            window.rfind(", "),
            window.rfind(" "),
        ]
        cut = max(candidates)
        if cut < max_chars // 2:
            cut = max_chars
        else:
            cut += 1
        chunks.append(remaining[:cut].strip())
        remaining = remaining[cut:].strip()
    if remaining:
        chunks.append(remaining)
    return chunks

units = []
max_payload_chars = MAX_BATCH_CHARACTERS - 80
unit_number = 0
for direction, records in source_partitions.items():
    for record in records:
        for source_field, target_field in FIELD_MAP[direction]:
            source_text = record[source_field]
            if not source_text:
                continue
            parts = split_long_text(source_text, max_payload_chars)
            for part_index, part_text in enumerate(parts):
                unit_number += 1
                units.append({
                    "unit_id": f"u{unit_number:08d}",
                    "record_id": record["id"],
                    "source_row_index": record["source_row_index"],
                    "direction": direction,
                    "source_field": source_field,
                    "target_field": target_field,
                    "part_index": part_index,
                    "part_count": len(parts),
                    "source_text": part_text,
                    "source_sha256": hashlib.sha256(part_text.encode("utf-8")).hexdigest(),
                })

write_jsonl_atomic(UNITS_PATH, units)
print(f"Unités créées : {len(units):,}")
print(f"Fichier         : {UNITS_PATH}")

Unités créées : 72,982
Fichier         : /home/vedem_worker/scrapt-bau/data/02b_text_browser/translation_units.jsonl


## 2. Regrouper les unités en blocs identifiables

Chaque fragment commence par un identifiant numérique de huit chiffres. Une traduction n'est acceptée que si tous les identifiants du bloc sont encore présents dans le résultat.

In [4]:
def serialize_unit(unit):
    numeric_id = unit["unit_id"][1:]
    return f"{numeric_id}: {unit['source_text']}"

batches = []
current_units = []
current_parts = []
current_length = 0

def flush_batch():
    global current_units, current_parts, current_length
    if not current_units:
        return
    batch_number = len(batches) + 1
    source_text = "\n".join(current_parts)
    batches.append({
        "batch_id": f"batch_{batch_number:05d}",
        "unit_ids": [unit["unit_id"] for unit in current_units],
        "source_text": source_text,
        "source_characters": len(source_text),
        "source_sha256": hashlib.sha256(source_text.encode("utf-8")).hexdigest(),
    })
    current_units, current_parts, current_length = [], [], 0

for unit in units:
    serialized = serialize_unit(unit)
    separator_length = 1 if current_parts else 0
    if current_parts and current_length + separator_length + len(serialized) > MAX_BATCH_CHARACTERS:
        flush_batch()
    current_units.append(unit)
    current_parts.append(serialized)
    current_length += separator_length + len(serialized)
flush_batch()

assert all(batch["source_characters"] <= MAX_BATCH_CHARACTERS for batch in batches)
assert sum(len(batch["unit_ids"]) for batch in batches) == len(units)
write_jsonl_atomic(BATCHES_PATH, batches)

total_characters = sum(batch["source_characters"] for batch in batches)
print(f"Blocs préparés       : {len(batches):,}")
print(f"Caractères avec identifiants : {total_characters:,}")
print(f"Blocs par session    : {min(BATCHES_PER_SESSION, len(batches))}")
print(f"Durée d'attente minimale par session : {BATCHES_PER_SESSION * DELAY_SECONDS / 60:.1f} minutes")

Blocs préparés       : 4,588
Caractères avec identifiants : 15,120,378
Blocs par session    : 100
Durée d'attente minimale par session : 13.3 minutes


## 3. Installer Playwright

Exécutez cette cellule dans un environnement possédant une interface graphique. Elle installe la bibliothèque Python puis le Chromium compatible géré par Playwright. L'installation du navigateur est ignorée automatiquement s'il est déjà présent.

In [5]:
try:
    import playwright
    print("Playwright est déjà installé.")
except ImportError:
    print("Installation de Playwright…")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "playwright"])
    import playwright
    print("Playwright installé.")

if BROWSER_CHANNEL is None:
    print("Vérification/installation du navigateur Chromium compatible…")
    subprocess.check_call([sys.executable, "-m", "playwright", "install", "chromium"])
    print("Chromium Playwright est prêt.")

Playwright est déjà installé.
Vérification/installation du navigateur Chromium compatible…
Chromium Playwright est prêt.


In [6]:
import os
from pathlib import Path

local_libs = (
    Path.cwd()
    / ".playwright-libs"
    / "usr"
    / "lib"
    / "x86_64-linux-gnu"
)

previous = os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = (
    str(local_libs) + (":" + previous if previous else "")
)

print("Bibliothèques Chromium :", local_libs)
print("Existe :", local_libs.exists())

Bibliothèques Chromium : /home/vedem_worker/scrapt-bau/.playwright-libs/usr/lib/x86_64-linux-gnu
Existe : True


## 4. Exécuter une session dans un navigateur visible

Avec `RUN_BROWSER = True`, le navigateur s'ouvre sur Google Traduction. Acceptez ou refusez les cookies si nécessaire, vérifiez visuellement **Français → Baoulé**, puis revenez au notebook et appuyez sur Entrée. Chaque exécution traite au maximum `BATCHES_PER_SESSION` blocs encore manquants.

Ne touchez plus à la page pendant la session. Chaque bloc est transmis par une URL `text=` puis lu directement dans le panneau de résultat. Le notebook sauvegarde immédiatement chaque succès et s'arrête au premier signe de protection ou si les identifiants numériques ne sont pas restitués correctement.

In [ ]:
class ProtectionDetected(RuntimeError):
    pass

def append_jsonl(path, record):
    with path.open("a", encoding="utf-8", newline="\n") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()

def completed_batch_ids():
    if not RESULTS_PATH.exists():
        return set()
    return {record["batch_id"] for record in read_jsonl(RESULTS_PATH) if record.get("status") == "ok"}

def expected_markers(batch):
    return [f"{unit_id[1:]}:" for unit_id in batch["unit_ids"]]

async def read_translation_from_page(page, batch):
    deadline = time.monotonic() + TRANSLATION_TIMEOUT_SECONDS
    marker_names = expected_markers(batch)
    while time.monotonic() < deadline:
        page_text = (await page.locator("body").inner_text()).lower()
        protection_terms = ("captcha", "trafic inhabituel", "unusual traffic", "too many requests", "erreur 429")
        if any(term in page_text for term in protection_terms):
            raise ProtectionDetected("Google a affiché une protection ou une limitation de trafic.")

        output = page.locator(f'[lang="{TARGET_LANGUAGE_CODE}"][jsname="jqKxS"]').first
        if await output.count() and await output.is_visible():
            translated = (await output.inner_text()).strip()
            if translated and all(marker in translated for marker in marker_names):
                return translated
        await page.wait_for_timeout(750)
    diagnostic_path = STEP_DIR / f"diagnostic_{batch['batch_id']}.png"
    await page.screenshot(path=str(diagnostic_path), full_page=True)
    raise TimeoutError(f"Traduction non récupérée ou numéros perdus. Capture : {diagnostic_path}")

async def run_browser_session():
    from playwright.async_api import async_playwright

    done = completed_batch_ids()
    pending_batches = [batch for batch in batches if batch["batch_id"] not in done]
    session_batches = pending_batches[:BATCHES_PER_SESSION]
    if not session_batches:
        print("Tous les blocs sont déjà terminés.")
        return
    print(f"Progression avant session : {len(done):,}/{len(batches):,} blocs terminés")
    print(f"Cette session traitera au maximum {len(session_batches)} blocs")

    async with async_playwright() as manager:
        browser_environment = os.environ.copy()
        if LOCAL_BROWSER_LIB_DIR.exists():
            existing_library_path = browser_environment.get("LD_LIBRARY_PATH", "")
            browser_environment["LD_LIBRARY_PATH"] = str(LOCAL_BROWSER_LIB_DIR) + (":" + existing_library_path if existing_library_path else "")
        launch_options = {"headless": False, "env": browser_environment}
        if BROWSER_CHANNEL:
            launch_options["channel"] = BROWSER_CHANNEL
        browser = await manager.chromium.launch(**launch_options)
        context = await browser.new_context(permissions=["clipboard-read", "clipboard-write"], locale="fr-FR")
        page = await context.new_page()
        await page.goto(
            f"https://translate.google.com/?hl=fr&sl=fr&tl={TARGET_LANGUAGE_CODE}&op=translate",
            wait_until="domcontentloaded",
            timeout=60_000,
        )
        input("Vérifiez Français → Baoulé dans le navigateur, puis appuyez sur Entrée ici… " )

        try:
            for position, batch in enumerate(session_batches, start=1):
                print(f"[{position}/{len(session_batches)}] {batch['batch_id']} — {batch['source_characters']} caractères")
                query_url = "https://translate.google.com/?" + urlencode({
                    "hl": "fr",
                    "sl": "fr",
                    "tl": TARGET_LANGUAGE_CODE,
                    "text": batch["source_text"],
                    "op": "translate",
                })
                await page.goto(query_url, wait_until="domcontentloaded", timeout=60_000)
                translated_text = await read_translation_from_page(page, batch)
                append_jsonl(RESULTS_PATH, {
                    "batch_id": batch["batch_id"],
                    "status": "ok",
                    "translated_text": translated_text,
                    "source_sha256": batch["source_sha256"],
                    "translated_at_utc": datetime.now(timezone.utc).isoformat(),
                    "engine": "google_translate_text_ui",
                    "target_language_code": TARGET_LANGUAGE_CODE,
                })
                print("  ✓ sauvegardé")
                await page.wait_for_timeout(int(DELAY_SECONDS * 1000))
        except ProtectionDetected as exc:
            print("ARRÊT DE SÉCURITÉ :", exc)
        except Exception as exc:
            print(f"ARRÊT SUR ERREUR : {type(exc).__name__}: {exc}")
        finally:
            await browser.close()
            completed_now = completed_batch_ids()
            print(f"Progression sauvegardée : {len(completed_now):,}/{len(batches):,} blocs terminés")
            print(f"Blocs restants : {len(batches) - len(completed_now):,}")

def run_coroutine_in_thread(coroutine):
    state = {}
    def target():
        try:
            state["result"] = asyncio.run(coroutine)
        except BaseException as exc:
            state["error"] = exc
    thread = threading.Thread(target=target)
    thread.start()
    thread.join()
    if "error" in state:
        raise state["error"]
    return state.get("result")

if RUN_BROWSER:
    run_coroutine_in_thread(run_browser_session())
else:
    print("Navigateur non lancé. Passez RUN_BROWSER à True dans la première cellule après avoir lu les consignes.")

## 5. Parser et contrôler les résultats disponibles

Cette cellule accepte les résultats partiels : si seuls trois blocs ont réussi, ils seront analysés et sauvegardés sans perdre la progression.

In [ ]:
unit_by_id = {unit["unit_id"]: unit for unit in units}
marker_pattern = re.compile(
    r"^[ \t]*(\d{8}):[ \t]*(.*?)(?=^[ \t]*\d{8}:|\Z)",
    flags=re.DOTALL | re.MULTILINE,
)

parsed_parts = {}
parse_errors = []
results = read_jsonl(RESULTS_PATH) if RESULTS_PATH.exists() else []
for result in results:
    if result.get("status") != "ok":
        continue
    matches = marker_pattern.findall(result["translated_text"])
    expected_ids = set(next(batch for batch in batches if batch["batch_id"] == result["batch_id"])["unit_ids"])
    matches = [(f"u{numeric_id}", text) for numeric_id, text in matches]
    found_ids = {unit_id for unit_id, _ in matches}
    if found_ids != expected_ids:
        parse_errors.append({"batch_id": result["batch_id"], "expected": sorted(expected_ids), "found": sorted(found_ids)})
        continue
    for unit_id, translated_text in matches:
        parsed_parts[unit_id] = translated_text.strip()

grouped = defaultdict(list)
for unit_id, translated_text in parsed_parts.items():
    unit = unit_by_id[unit_id]
    key = (unit["record_id"], unit["direction"], unit["target_field"])
    grouped[key].append((unit["part_index"], unit["part_count"], translated_text, unit["source_text"]))

partial_records = []
for (record_id, direction, target_field), parts in grouped.items():
    parts.sort(key=lambda item: item[0])
    expected_part_count = parts[0][1]
    complete = len(parts) == expected_part_count and [item[0] for item in parts] == list(range(expected_part_count))
    translated_text = " ".join(item[2] for item in parts) if complete else None
    source_text = " ".join(item[3] for item in parts) if complete else None
    partial_records.append({
        "record_id": record_id,
        "direction": direction,
        "target_field": target_field,
        "complete": complete,
        "translated_text": translated_text,
        "unchanged_from_source": complete and translated_text == source_text,
        "part_count": expected_part_count,
    })

write_jsonl_atomic(PARTIAL_TRANSLATIONS_PATH, partial_records)
print(f"Résultats navigateur : {len(results)} blocs")
print(f"Fragments analysés   : {len(parsed_parts)}")
print(f"Champs complets      : {sum(r['complete'] for r in partial_records)}")
print(f"Erreurs de balises   : {len(parse_errors)}")
print(f"Traductions inchangées : {sum(r['unchanged_from_source'] for r in partial_records)}")
print("Sortie partielle :", PARTIAL_TRANSLATIONS_PATH)

## Critères avant toute montée en charge

Le pilote n'est validé que si les dix blocs sont récupérés sans protection, si aucune balise n'est perdue, si les textes ne sont pas tronqués et si un locuteur baoulé juge un échantillon suffisamment fidèle. Le passage à plus de dix blocs doit rester progressif et respecter les conditions du service.